In this notebook we'll look to forecast the number of airline passengers. We'll take a Bayesian approach and implement the main logic behind Facebook's [Prophet](https://peerj.com/preprints/3190/) model for time series analysis using [PyMC](https://www.pymc.io/welcome.html).

In [ ]:
!pip install aesara -q
!pip install pymc -q

In [ ]:
import aesara
import arviz
import matplotlib.pyplot as plt
import numpy
import pandas
import plotly.express as px
import plotly.graph_objects as go
import pymc
import scipy

## Exploratory Analysis

Let's open our dataset and look at the first few samples

In [ ]:
df = pandas.read_csv("/kaggle/input/air-passengers/AirPassengers.csv")
df.head()

Let's rename our columns and cast our time elements to a datetime object

In [ ]:
df.columns = ["Datetime", "Passengers"]
df["Datetime"] = pandas.to_datetime(df["Datetime"], format="%Y-%m")
df = df.set_index("Datetime")
df.head()

If we visualize our passenger count over time, we'll see that it has a trend as well as a seasonal component with about three local peaks on an annual basis:

In [ ]:
px.line(df, y="Passengers").update_layout(showlegend=False)

Since the magnitude of the seasonality is related to the underlying trend we are likely looking at a multiplicative process where the number of passengers is given by

$$\rm
Passengers(t) \sim Trend(t) \times Seasonality(t) \times Residual_{t}
$$

This can be converted to an additive process by taking the logarithm

$$\rm
\log{(Passengers)} \sim \log{(Trend)} + \log{(Seasonality)} + \log{(Residual)}
$$

which results in a more homoscedastic variance after applying a differencing operation, as illustrated in the figures below.

In [ ]:
labels = dict(x="Datetime", y=r"$y_t - y_{t-1}$")

px.line(
    x=df.index,
    y=df["Passengers"].diff(periods=1),
    labels=labels
).update_layout(title="Original Data: Heteroscedastic").show()

px.line(
    x=df.index,
    y=numpy.log(df["Passengers"]).diff(periods=1),
    labels=labels
).update_layout(title="Log Transformation: Homoscedastic").show()

There are three repeating peaks in our difference series: December (Christmas + other Holidays), March (Spring break + vacations), and June/July (vacations + no school). October also has a few minor peaks that occur occassionally.

Constructing a periodogram we see there is a strong yearly seasonality (as expected) while our recurring peaks lead to a strong semi-annual periodicity as well as a few weaker ones on a more frequent basis.

In [ ]:
f, P = scipy.signal.periodogram(numpy.log(df["Passengers"]).diff(1)[1:], fs=12)
px.line(x=f, y=P**0.5, labels=dict(x="Frequency (1/Year)", y="Power Spectrum"))

## Prophet Time Series Model

Prophet decomposes a time series using a generalized additive model

$$
y(t) = g(t) + s(t) + h(t) + \epsilon_{t}
$$

where $g(t)$ is the trend, $s(t)$ the seasonality, $h(t)$ the effect of holidays and other events, and $\epsilon_{t}$ represents the errors / residuals (which are assumed to be normally distributed). We'll break down these components in the following sections.

Before we get ahead of ourselves let's split our dataset into a train and test set. We'll use 1959 as the cutoff year so that we have two years for evaluating our forecast model. Let's normalize our training and testing times so that the series starts at $t=0$ and reaches $t=1$ at the end of our training dataset and log transform our passenger data.

In [ ]:
timesplit = pandas.Timestamp("1959-01-01")

# Split into test and train dataset
data = {
    "train": df[df.index < timesplit],
    "test": df[df.index >= timesplit]
}

# Normalize time so that the training time is between 0 and 1
time_scaled = {
    k: (data[k].index - data["train"].index[0]).days / (data["train"].index[-1] - data["train"].index[0]).days
    for k in ["train", "test"]
}

# Log transform passenger data
y_log = {
    k: numpy.log(data[k]["Passengers"].to_numpy())
    for k in ["train", "test"]
}

### Trend

Prophet implements two trends that capture most time series. The first is a linear model

$$
g(t) = m + k\cdot t
$$

where $m$ is the intercept and $k$ the growth rate. In general trends will not continue on indefinetely but reach some carrying capacity. Prophet models these nonlinear cases with a logistic growth model,

$$
g(t) = \frac{C}{1 + e^{-k \cdot (t - m)}}
$$

where $C$ is the carrying capacity, $k$ the growth rate, and $m$ is an offset parameter. 

Growth rates are not always constant and the capacity can change with time as well. To account for these changes, Prophet allows a user to specify a time varying capacity while utilizing changepoints internally to model changing growth rates. 

#### Changepoints

Consider a set of $S$ changepoints located at $s_{j}$ for $j = 1, \ldots, S$. Each changepoint $s_j$ will have a rate adjustment $\delta_j$ associated with it such that the growth rate at time $t$ is the original growth rate $k_0$ plus the sum of the previous changes

$$
k(t) = k_0 + \sum_{j:t>s_j} \delta_j
$$

If we define a vector $\boldsymbol{a}(t) \in \{0,1\}^{S}$ such that

$$
a_j(t) = \left \lbrace \begin{array}{ll}
1, & \text{if } t \ge s_j \\
0, & \text{otherwise}
\end{array} \right.
$$

then the growth rate can also be expressed as

$$
k(t) = k_0 + \boldsymbol{a}(t)^{T} \delta
$$

Prophet's philosophy is to create a large number of changepoints while assigning a sparse prior $\delta_j \sim \mathrm{Laplace}(0, \tau)$ to the rate adjustments, which is similar to Lasso regression. 

When forecasting Prophet makes inferences from previous changepoints to estimate future uncertainties rather than rely on the last growth rate. We won't concern ourselves with this aspect here, but may return to it in the future.

#### Logistic Growth with Changepoints

Introducing changepoints requires some corrections to our offset parameter $m$ so that endpoints of the segments are connected. For the logistic growth model these corrections are

$$
\gamma_j = \left(s_j - m - \sum_{l<j} \gamma_l \right)\left(1 + \frac{k_0 + \sum_{l<j}\delta_l}{k_0 + \sum_{l \le j} \delta_l}\right)
$$

and our model becomes

$$
g(t) = \frac{C(t)}{1 + \exp[-(k_0 + \boldsymbol{a}(t)^{T}\boldsymbol{\delta})(t - (m + \boldsymbol{a}(t)^{T} \boldsymbol{\gamma})]}
$$

#### Linear Trend with Changepoints

With changepoints our linear trend becomes the piecewise model

$$
g(t) = (k_0 + \boldsymbol{a}(t)^{T} \boldsymbol{\delta})\cdot t + (m + \boldsymbol{a}(t)^{T} \boldsymbol{\gamma})
$$

where our offset corrections are

$$
\gamma_j = -s_j \delta_j
$$

### Seasonality
Prophet uses a Fourier series to model the seasonality,

$$
s(t) = \sum_{n=1}^{N} a_n \sin\left(\frac{2\pi n t}{P}\right) + b_n \cos\left(\frac{2\pi n t}{P}\right)
$$

where $n$ is the mode, $N$ the total number of modes, and $P$ a reference period in days (e.g. 7 days for a week and 365.25 days for a year). Let's set our period to 365.25 days and create a mode to represent the first two peaks in the Periodogram (using five modes can be seen as an upper limit in our dataset, as including more results in moving beyond the Nyquist frequency) and try to capture the additional flucutations with our Holiday trends.

In [ ]:
# Frequency
ωt = {
    k: 2*numpy.pi*data[k].index.day_of_year / 365.25
    for k in ["train", "test"]
}

# Fourier terms
n_modes = 2
fourier_series = {
    k: pandas.DataFrame({
        f"{func}_{mode}": getattr(numpy, func)(ωt[k] * mode)
        for mode in numpy.arange(1,n_modes+1)
        for func in ("sin", "cos")
    })
    for k in ["train", "test"]
}

fourier_series["train"].head()

We'll represent the Fourier coefficients as $\boldsymbol{\beta} = [a_1, b_1, \cdots, a_n, b_n]^{T}$. To regularize these coefficients, Prophet utilizes a smoothing prior $\boldsymbol{\beta} \sim \mathrm{Normal}(0,\sigma^2)$ that is analogous to ridge regression.

### Holidays and other Events

Prophet uses a set of indicator functions for modeling holidays and other events. Each holiday $i$ has associated with it a set of dates $D_{i}$ and a parameter $\kappa_i$ to represent the change in the forecast. Prophet then constructs a matrix of regressors

$$
Z(t) = [\boldsymbol{1}(t \in D_1), \ldots, \boldsymbol{1}(t \in D_L)
$$

and the holiday contribution is given by

$$
h(t) = Z(t) \boldsymbol{\kappa}
$$

Prophet assigns a prior $\boldsymbol{\kappa} \sim \mathrm{Normal}(0,\nu^2)$.

Since our data is binned by month, it may be more difficult to identify the effects of holidays and other important events so we'll want to keep the number to a minimum. Let's create one for March to represent Spring break and one for December to represent Christmas and the other holidays around then.

In [ ]:
holidays = {"Spring Break": 3, "Christmas": 12}

is_holiday = {
    k: pandas.DataFrame({
        holiday: numpy.where(data[k].index.month == month, 1, 0)
        for holiday, month in holidays.items()
    }) for k in ["train", "test"]
}

is_holiday["train"].head(12).T

### Model

Let's go ahead and create our model and generate samples from the prior distribution

In [ ]:
coords = dict({
    "fourier_coef": numpy.arange(n_modes*2),
    "changepoints": numpy.arange(10),
    "holidays": list(holidays.keys()),
})

linear_trend:bool = False # Whether to use linear (True) or logistic growth (False)

with pymc.Model(coords=coords) as prophet:
    # ====================================================
    # Used data. Note that we'll make them mutable so that
    # we can replace them with our testing data later.
    # ====================================================
    time = pymc.MutableData("time", time_scaled["train"].to_numpy())
    ft = pymc.MutableData("fourier_series", fourier_series["train"].to_numpy())
    hday = pymc.MutableData("hday", is_holiday["train"].to_numpy())
    ylog = pymc.MutableData("ylog", y_log["train"])
    
    # ====================================================
    # Trend Component
    # ====================================================
    if linear_trend:
        k0 = pymc.Normal(name="k0", mu=0, sigma=2) # Growth rate
        m = pymc.Normal(name="m", mu=4.75, sigma=1) # Intercept
    else:   
        C = pymc.TruncatedNormal(name="C", mu=10, sigma=5, lower=0) # Capacity (assume constant)
        k0 = pymc.Normal(name="k0", mu=0, sigma=2) # Growth rate
        m = pymc.Normal(name="m", mu=0.5, sigma=0.5) # Offset
    
    # changepoints
    mu = numpy.linspace(0.1, 0.8, len(coords["changepoints"]))
    s = pymc.MutableData("s", mu, dims="changepoints")

    # change rate
    δ = pymc.Laplace(name="δ", mu=0, b=0.1, dims="changepoints")
    a = pymc.Deterministic("a", var=pymc.math.where(time[:,None] < s[None,:], numpy.array(0.), numpy.array(1.)))

    # trend
    if linear_trend:
        γ = pymc.Deterministic("γ", var=-s*δ, dims="changepoints")
        func = (k0 + pymc.math.dot(a,δ))*time + (m + pymc.math.dot(a,γ))
    else:
        # To calculate γ, we'll need to use a loop, which can
        # be accomplished with the aesara.scan method. We'll
        # need to iterate over s and δ, return γ, keep track of
        # the cumulative sum of γ and δ, and also access m and k0.
        # These then become the inputs to our loop function,
        # with the sequences first, outputs, followed by non-sequences
        def one_step(sj, δj, γ, γ_cumsum, δ_cumsum, m, k):
            left = sj - m - γ_cumsum
            right = 1 - (k + δ_cumsum) / (k + δ_cumsum + δj)
            γj = left * right
            return γj, γ_cumsum + γj, δ_cumsum + δj
        
        # Our initial γ_cumsum and δ_cumsum are zero. We can 
        # initialize γ to be zero as well as a placeholder.
        zero = aesara.tensor.as_tensor(0., dtype=s.dtype)
        output, updates = aesara.scan(
            fn=one_step,
            sequences=[s, δ],
            outputs_info = [zero for _ in range(3)],
            non_sequences=[m, k0],
        )
        
        # The first output corresponds to gamma. Note that aesara.scan
        # will return a sequence corresponding to each of the outputs,
        # so output[0] will give us all the calculated γ values
        γ = pymc.Deterministic(
            name="γ",
            var=output[0],
            dims="changepoints"
        )

        func = C / (1 + pymc.math.exp(-(k0 + pymc.math.dot(a,δ))*(time - (m+pymc.math.dot(a,γ)))))

    trend = pymc.Deterministic(
        name="trend",
        var = func
    )

    # ====================================================
    # Seasonal Component
    # ====================================================
    β = pymc.Normal(name="β", mu=0, sigma=0.3, dims="fourier_coef") # smoothing prior (ridge)
    seasonal = pymc.Deterministic(
        name="seasonal",
        var=pymc.math.dot(ft, β)
    )
    
    # ====================================================
    # Holidays and other Events
    # ====================================================
    κ = pymc.Normal(name="κ", mu=0, sigma=0.3, dims="holidays")
    holiday = pymc.Deterministic(name="holiday", var=pymc.math.dot(hday, κ))
    
    # ====================================================
    # Likelihood
    # ====================================================
    sigma = pymc.HalfNormal("error", sigma=0.1)
    pymc.Normal(
        name="likelihood",
        mu=trend + seasonal + holiday,
        sigma=sigma,
        observed=ylog,
    )

    # ====================================================
    # Prior distribution
    # ====================================================
    prior = pymc.sample_prior_predictive(samples=100)
    
pymc.model_to_graphviz(prophet)

Our $\gamma$ calculation is a bit involved, so let's do a quick test to make sure we are getting the expected values

In [ ]:
def get_γ(s, δ, m, k):
    γ = []
    for j in range(len(δ)):
        left = (s[j] - m - sum(γ))
        right = 1 - (k + sum(δ[:j])) / (k + sum(δ[:j+1]))
        γ.append(left * right)
    return numpy.asarray(γ, dtype='float32')
        
print(output[0].eval())
print(get_γ(s.eval(), δ.eval(), m.eval(), k0.eval()))

Looks like our $\gamma$ calculation is working correctly.

Let's plot our prior distributions to make sure that we have reasonable starting values.

In [ ]:
from plotly.subplots import make_subplots

n = 100
x = data["train"].index

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Full Model","Trend", "Seasonality"))

# Likelihood
kwargs = {
    'marker': {'color': px.colors.qualitative.Plotly[0]},
    'opacity': 0.1
}

for i in range(100):
    y = arviz.extract_dataset(prior, group="prior_predictive").sel(draw=i, chain=0)["likelihood"]
    fig.add_trace(go.Scatter(x=x, y=y, **kwargs), 1, 1)

    y = arviz.extract_dataset(prior, group="prior").sel(draw=i, chain=0)["trend"]
    fig.add_trace(go.Scatter(x=x, y=y, **kwargs), 2, 1)

    y = arviz.extract_dataset(prior, group="prior").sel(draw=i, chain=0)["seasonal"]
    fig.add_trace(go.Scatter(x=x, y=y, **kwargs), 3, 1)

fig.add_trace(go.Scatter(x=x, y=y_log["train"], mode="lines", marker=dict(color="black")), 1, 1)
fig.add_trace(go.Scatter(x=x, y=y_log["train"], mode="lines", marker=dict(color="black")), 2, 1)
fig.add_trace(go.Scatter(x=x, y=y_log["train"] - pandas.DataFrame(y_log["train"]).rolling(6).mean().values.reshape(-1), mode="lines", marker=dict(color="black")), 3, 1)

fig.update_xaxes(title_text="Datetime", row=3, col=1)
fig.update_yaxes(title_text="Log(Passengers)", row=1, col=1)
fig.update_yaxes(title_text="Log(Passengers)", row=2, col=1)
fig.update_layout(height=800, showlegend=False)
fig.show()

The full model and trend may be a bit wider than needed, but the spread doesn't look too egregious.

Let's go ahead and run our MCMC method. Note that using the linear model will run much faster, which could be something worth looking into.

In [ ]:
with prophet:
    trace = pymc.sample(draws=500, tune=500, chains=3, target_accept=0.9)
    posterior = pymc.sample_posterior_predictive(trace=trace)

Let's go ahead and visualize our posterior distributions

In [ ]:
from plotly.subplots import make_subplots

n = 200
x = data["train"].index
kwargs = {'marker': {'color': px.colors.qualitative.Plotly[0]}, 'opacity': 0.1}

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Posterior Predictive","Posterior Trend", "Posterior Seasonality"))

yp = numpy.exp(arviz.extract_dataset(posterior, group="posterior_predictive", num_samples=n)["likelihood"])
yt = numpy.exp(arviz.extract_dataset(trace, group="posterior", num_samples=n)["trend"])
ys = numpy.exp(arviz.extract_dataset(trace, group="posterior", num_samples=n)["seasonal"])

for i in range(n):
    fig.add_trace(go.Scatter(x=x, y=yp[:,i], **kwargs), 1, 1)
    fig.add_trace(go.Scatter(x=x, y=yt[:,i], **kwargs), 2, 1)
    fig.add_trace(go.Scatter(x=x, y=ys[:,i], **kwargs), 3, 1)

fig.add_trace(go.Scatter(x=x, y=numpy.exp(y_log["train"]), mode="lines", marker=dict(color="black")), 1, 1)
fig.add_trace(go.Scatter(x=x, y=numpy.exp(y_log["train"]), mode="lines", marker=dict(color="black")), 2, 1)

fig.update_xaxes(title_text="Datetime", row=3, col=1)
fig.update_yaxes(title_text="Passengers", row=1, col=1)
fig.update_yaxes(title_text="Passengers", row=2, col=1)
fig.update_layout(height=800, showlegend=False)
fig.show()

Our model looks like it is doing a fairly decent job, although it starts to underestimate the number of passengers near the beginning of the year towards the end of time series.

Let's go ahead and plot the posterior distributions for our parameters

In [ ]:
var_names = ["k0", "m", "δ", "β", "κ"]
if not linear_trend:
    var_names = ["C", *var_names]
arviz.plot_trace(trace, var_names=var_names, legend=True, figsize=(12,20))
plt.tight_layout();

Our capacity has an exteneded tail, likely indicating that it is growing with time, so that could be an area to improve our model. Our changepoints aren't all that sparse either, so perhaps a greater regularization parameter would be helpful.

### Forecasting

Let's go ahead and see how well our model does on our testing dataset. We'll need to update our data and then we can sample the posterior predictive.

In [ ]:
with prophet:
    pymc.set_data({
        "time": time_scaled["test"].to_numpy(),
        "fourier_series": fourier_series["test"].to_numpy(),
        "hday": is_holiday["test"].to_numpy(),
        "ylog": y_log["test"],
    })
    
    test_posterior = pymc.sample_posterior_predictive(trace=trace)

Let's visualize our forecast

In [ ]:
n = 200
x = data["test"].index
y_pred = numpy.exp(arviz.extract_dataset(test_posterior, "posterior_predictive", num_samples=n)["likelihood"])
y_test = data["test"]["Passengers"]
kwargs = {'marker': {'color': px.colors.qualitative.Plotly[0]}, 'opacity': 0.1}

fig = go.Figure()

for i in range(n):
    fig.add_trace(go.Scatter(x=x, y=y_pred[:,i], name="Forecast", **kwargs))
fig.add_trace(go.Scatter(x=x, y=y_test, mode="lines", marker=dict(color="black")))
fig.update_layout(xaxis_title="Datetime", yaxis_title="Passengers", title="Passenger Forecast", showlegend=False)
fig.show()

Looks like our model is tending to underforecast the number of passengers. This could be tied to an increasing capacity with time or an underestimate of the growth near the end, particularly as there was a noticeable underestimation of the number of passengers during the first few months of 1958.

To evalue our forecast, let's go ahead and compute the mean absolute percentage error.

In [ ]:
x = (data["test"].index - data["test"].index[0]).days
y_test = y_test.values.reshape(-1,1)

dy = 100 * abs(y_pred - y_test).values / y_test
dy_mu = dy.mean(-1)
dy_sd = dy.std(-1, ddof=1)

fig = go.Figure()
fig.add_trace(go.Scatter(name="Offset", x=x, y=dy_mu))
fig.add_trace(go.Scatter(name="Upper Bound", x=x, y=dy_mu + 2*dy_sd, mode="lines", marker=dict(color="#444"), line=dict(width=0)))
fig.add_trace(go.Scatter(name="Lower Bound", x=x, y=dy_mu - 2*dy_sd, mode="lines", marker=dict(color="#444"), line=dict(width=0), fill='tonexty', fillcolor='rgba(68, 68, 68, 0.3)'))
fig.update_layout(xaxis_title="Forecast Horizon (Days)", yaxis_title="Mean Absolute % Error", showlegend=False)
fig.show()

Our mean offset is about 10% and the observed data is close to within about two standard deviations. There is a bit of a an increased error with time, but it isn't too much of a trend so our model is doing a fairly decent job at capturing the underlining trends.

## Conclusion

In this notebook we looked at airline passenger data from 1949 - 1960 and performed a time series analysis using the main logic behind Facebook's Prophet model, decomposing the series into trend, seasonality, and holiday components. Our model was able to provide a forecast to within about 10% two years into the future, but tended to underestimate the observed number of passengers.